In [1]:
import sys
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.model_selection import train_test_split

sys.path.append(r'G:\DATASCIENCE\End_to_End_Project\Banking Fraud Detection')
from src.fraud_utils import (
    engineer_features, engineer_features_xgb,
    CAT_COLS, BINARY_COLS_LR, BINARY_COLS_XGB,
    PASSTHROUGH_COLS_XGB, LOG_SCALE_COLS_LR, SCALE_ONLY_COLS_LR
)

os.chdir(r'G:\DATASCIENCE\End_to_End_Project\Banking Fraud Detection')
os.makedirs('data',   exist_ok=True)
os.makedirs('models', exist_ok=True)


In [2]:
DATA_PATH = r'G:\DATASCIENCE\End_to_End_Project\Banking Fraud Detection\bank_fraud.csv'
df = pd.read_csv(DATA_PATH)

assert df.shape == (1000000, 26), f"Shape mismatch: {df.shape}"
assert df['is_fraud'].sum() == 55255, f"Fraud count mismatch: {df['is_fraud'].sum()}"
assert round(df['is_fraud'].mean(), 4) == 0.0553, f"Fraud rate mismatch: {df['is_fraud'].mean()}"

print(f"Shape: {df.shape} | Fraud: {df['is_fraud'].sum():,} | Rate: {df['is_fraud'].mean():.4f}")
print(df.columns.tolist())


Shape: (1000000, 26) | Fraud: 55,255 | Rate: 0.0553
['transaction_id', 'customer_id', 'transaction_date', 'transaction_time', 'hour_of_day', 'is_weekend', 'is_night_transaction', 'country', 'city', 'merchant_category', 'payment_method', 'device_type', 'customer_age', 'credit_score', 'account_age_years', 'account_balance', 'transaction_amount', 'num_prev_transactions', 'transaction_freq_monthly', 'distance_from_home_km', 'time_since_last_txn_hrs', 'is_international', 'failed_attempts', 'pin_changed_recently', 'is_fraud', 'fraud_type']


In [3]:
cols_to_drop = [
    'transaction_id', 'customer_id', 'transaction_time',
    'fraud_type', 'transaction_date', 'city'
]

df.drop(columns=cols_to_drop, inplace=True)

assert df.shape[1] == 20, f"Expected 20 columns, got {df.shape[1]}"
assert 'fraud_type' not in df.columns
assert 'is_fraud' in df.columns

print(f"Shape: {df.shape}")
print(df.columns.tolist())


Shape: (1000000, 20)
['hour_of_day', 'is_weekend', 'is_night_transaction', 'country', 'merchant_category', 'payment_method', 'device_type', 'customer_age', 'credit_score', 'account_age_years', 'account_balance', 'transaction_amount', 'num_prev_transactions', 'transaction_freq_monthly', 'distance_from_home_km', 'time_since_last_txn_hrs', 'is_international', 'failed_attempts', 'pin_changed_recently', 'is_fraud']


In [4]:
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp
)

assert abs(y_train.mean() - 0.0553) < 0.001
assert abs(y_val.mean()   - 0.0553) < 0.001
assert abs(y_test.mean()  - 0.0553) < 0.001

joblib.dump((X_test, y_test), 'data/test_set.pkl')

print(f"Train:      {X_train.shape[0]:,} | fraud rate: {y_train.mean():.4f}")
print(f"Validation: {X_val.shape[0]:,} | fraud rate: {y_val.mean():.4f}")
print(f"Test:       {X_test.shape[0]:,} | fraud rate: {y_test.mean():.4f}")
print("Test set locked.")


Train:      699,975 | fraud rate: 0.0553
Validation: 150,025 | fraud rate: 0.0553
Test:       150,000 | fraud rate: 0.0553
Test set locked.


In [5]:
X_train_eng = engineer_features(X_train)
temp = X_train_eng.copy()
temp['is_fraud'] = y_train.values

for feature, val, low, high in [
    ('high_risk_attempts',  1, 0.13, 0.16),
    ('night_international', 1, 0.11, 0.14),
    ('high_risk_merchant',  1, 0.08, 0.095),
]:
    rate = temp.groupby(feature)['is_fraud'].mean()[val]
    assert low <= rate <= high, f"{feature}={val}: {rate:.4f} outside ({low}, {high})"
    print(f"{feature}={val}: {rate:.4f} ✅")


high_risk_attempts=1: 0.1466 ✅
night_international=1: 0.1220 ✅
high_risk_merchant=1: 0.0874 ✅


In [6]:
log_then_scale = Pipeline([
    ('log',   FunctionTransformer(np.log1p, validate=True)),
    ('scale', StandardScaler())
])

preprocessor_lr = ColumnTransformer(
    transformers=[
        ('log_scale',   log_then_scale,                                        LOG_SCALE_COLS_LR),
        ('scale',       StandardScaler(),                                      SCALE_ONLY_COLS_LR),
        ('ohe',         OneHotEncoder(handle_unknown='ignore',
                                      sparse_output=False),                    CAT_COLS),
        ('pass_binary', 'passthrough',                                         BINARY_COLS_LR),
    ],
    remainder='drop'
)

preprocessor_xgb = ColumnTransformer(
    transformers=[
        ('ohe',         OneHotEncoder(handle_unknown='ignore',
                                      sparse_output=False),                    CAT_COLS),
        ('pass_binary', 'passthrough',                                         BINARY_COLS_LR),
        ('pass_num',    'passthrough',                                         PASSTHROUGH_COLS_XGB),
    ],
    remainder='drop'
)


In [7]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
import time

pipeline_lr = Pipeline([
    ('engineer',     FunctionTransformer(engineer_features, validate=False)),
    ('preprocessor', preprocessor_lr),
    ('model',        LogisticRegression(
                         class_weight='balanced',
                         max_iter=1000,
                         random_state=42,
                         solver='lbfgs'
                     ))
])

pipeline_xgb = Pipeline([
    ('engineer',     FunctionTransformer(engineer_features, validate=False)),
    ('preprocessor', preprocessor_xgb),
    ('model',        XGBClassifier(
                         scale_pos_weight=17,
                         n_estimators=300,
                         learning_rate=0.1,
                         max_depth=6,
                         random_state=42,
                         eval_metric='aucpr',
                         n_jobs=-1
                     ))
])

pipeline_if = Pipeline([
    ('engineer',     FunctionTransformer(engineer_features, validate=False)),
    ('preprocessor', preprocessor_xgb),
    ('model',        IsolationForest(
                         contamination=0.055,
                         n_estimators=100,
                         random_state=42,
                         n_jobs=-1
                     ))
])

print("Fitting LR...")
t0 = time.time()
pipeline_lr.fit(X_train, y_train)
print(f"LR done — {time.time()-t0:.0f}s")

print("Fitting XGBoost...")
t0 = time.time()
pipeline_xgb.fit(X_train, y_train)
print(f"XGBoost done — {time.time()-t0:.0f}s")

print("Fitting Isolation Forest...")
t0 = time.time()
pipeline_if.fit(X_train)
print(f"IF done — {time.time()-t0:.0f}s")


Fitting LR...
LR done — 7s
Fitting XGBoost...
XGBoost done — 15s
Fitting Isolation Forest...
IF done — 6s


In [8]:
lr_probs_val  = pipeline_lr.predict_proba(X_val)[:, 1]
xgb_probs_val = pipeline_xgb.predict_proba(X_val)[:, 1]
if_scores_val = pipeline_if.decision_function(X_val)

assert lr_probs_val.shape  == (len(X_val),)
assert xgb_probs_val.shape == (len(X_val),)
assert if_scores_val.shape == (len(X_val),)
assert lr_probs_val.min()  >= 0 and lr_probs_val.max()  <= 1
assert xgb_probs_val.min() >= 0 and xgb_probs_val.max() <= 1

print(f"LR  — shape: {lr_probs_val.shape}  | range: [{lr_probs_val.min():.4f}, {lr_probs_val.max():.4f}]")
print(f"XGB — shape: {xgb_probs_val.shape} | range: [{xgb_probs_val.min():.4f}, {xgb_probs_val.max():.4f}]")
print(f"IF  — shape: {if_scores_val.shape}  | range: [{if_scores_val.min():.4f}, {if_scores_val.max():.4f}]")


LR  — shape: (150025,)  | range: [0.1715, 0.9724]
XGB — shape: (150025,) | range: [0.0033, 0.9057]
IF  — shape: (150025,)  | range: [-0.0993, 0.1104]


In [9]:
joblib.dump((X_train, y_train), 'data/train_set.pkl')
joblib.dump((X_val, y_val),     'data/val_set.pkl')
joblib.dump(pipeline_lr,        'models/pipeline_lr.pkl')
joblib.dump(pipeline_xgb,       'models/pipeline_xgb.pkl')
joblib.dump(pipeline_if,        'models/pipeline_if.pkl')

feature_names = engineer_features(X_train).columns.tolist()
joblib.dump(feature_names, 'models/feature_names.pkl')

print("Artifacts saved:")
for p in ['data/train_set.pkl', 'data/val_set.pkl', 'data/test_set.pkl',
          'models/pipeline_lr.pkl', 'models/pipeline_xgb.pkl',
          'models/pipeline_if.pkl', 'models/feature_names.pkl']:
    size = os.path.getsize(p) / 1e6
    print(f"  {p} — {size:.1f} MB")


Artifacts saved:
  data/train_set.pkl — 119.0 MB
  data/val_set.pkl — 25.5 MB
  data/test_set.pkl — 25.5 MB
  models/pipeline_lr.pkl — 0.0 MB
  models/pipeline_xgb.pkl — 1.3 MB
  models/pipeline_if.pkl — 1.2 MB
  models/feature_names.pkl — 0.0 MB
